In [1]:
import sympy as sp
from sympy import *
init_printing(use_unicode=True)
import random
import numpy as np
import math
import pandas as pd

# RKPW novo

In [2]:
def rkpw(Lambda, c):
    n = len(c)
    a = [1.0, Lambda[0]]
    b = [c[0]]
    for j in range(2, n + 1):
        lam_j = Lambda[j - 1]
        a.append(lam_j)
        b.append(0.0)
        delta1 = c[j - 1]
        delta2 = 0.0
        for i in range(1, j):
            if (b[i - 1] != 0.0 or delta1 !=0.0):
                r = sqrt(b[i-1]*b[i-1] + delta1*delta1) 
                gamma = b[i - 1]/r
                sigma = -delta1/r
                
                gammasqr = gamma*gamma
                sigmasqr = sigma*sigma
                gs = gamma*sigma
                
                b[i - 1] = r 
                delta1 = (gammasqr - sigmasqr)*delta2 + gs*(a[i] - a[j]) 
                a_j = a[j] 
                a[j] = a[j]*gammasqr + 2.0*gs*delta2 + a[i]*sigmasqr 
                a[i] = a[i]*gammasqr - 2.0*gs*delta2 + a_j*sigmasqr
                delta2 = b[i]*sigma 
                b[i] *= gamma 
        b[j - 1] = delta1   
    T = zeros(n) 
    for i in range(n):
        T[i, i] = a[i + 1]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i + 1]
    return T

# INVBI novo (Cramer)

In [3]:
def invbi_cramer(Lambda, beta):
    n = len(Lambda)
    a = [Lambda[0]]
    b = []
    w = [1.0]
    u = 1.0 
    for i in range(1, n): 
        alpha = beta[i - 1]*u 
        alpha2 = alpha*alpha
        lam_i = Lambda[i]
        B = [1]*i 
        for k in range(i - 2, -1, -1):
            B[k] = B[k + 1]*b[k]*b[k]
        b2_prod = B[0] 
        det = prod(Lambda[k] - lam_i for k in range(i)) 
        det2 = det*det 
        
        Da, Db = 0.0, 1.0 
        Dc = a[0] - lam_i 
        Sa = det2
        Sb = det2 + alpha2*b2_prod 
        
        a.append(lam_i)
        b.append(alpha) 
        a[0] -= alpha2*b2_prod*Dc/Sb 
        for j in range(1, i):
            Dc2 = Dc*Dc
            Sc = Sb + alpha2*B[j]*Dc2 
            
            Da, Db = Db, Dc
            aj_shift = a[j] - a[-1]         
            Dc = Db*aj_shift - Da*b[j - 1]*b[j - 1]
            
            inv_Sb = 1.0/Sb
            a[j] += alpha2*Db*B[j]*((aj_shift*Db - Dc)*inv_Sb - Dc/Sc) 
            b[j - 1] *= sqrt(Sc*Sa)*inv_Sb
            
            Sa, Sb = Sb, Sc
            
        a[i] += alpha2*det*Db/Sb
        b[i - 1] = alpha*sqrt(Sa*det2)/Sb 
        
        for j in range(len(w)):
            w[j] = beta[i - 1]*w[j]/(lam_i - Lambda[j])
        w.append(1)
        u = sqrt(sum(w[k]*w[k] for k in range(len(w)-1)) + 1.0)  
    T = zeros(n) 
    for i in range(n):
        T[i, i] = a[i]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i]
    return T

# INVBI (Gaussian)

In [4]:
def invbi_gauss(Lambda, beta): 
    n = len(Lambda)
    w = [1.0]
    u = 1.0
    a = [Lambda[0]]
    b = []
    for i in range(2, n + 1):
        alpha = beta[i - 2]*u
        Sd = a + [Lambda[i - 1]]
        Soff = b + [alpha]
        h = []
        f = []
        x = a[0] - Lambda[i - 1]
        h.append(x)
        for j in range(1, i - 1):
            y = b[j - 1] / h[j - 1]
            f.append(y)
            x = (a[j] - Lambda[i - 1]) - b[j - 1]*y
            h.append(x)
        v = [0.0] * (i - 1)
        v[-1] = 1.0/h[-1]
        for j in range(i - 3, -1, -1):
            v[j] = -f[j]*v[j+1]
        v = [val * alpha for val in v]
        n1 = 1.0
        n2 = n1 + v[0]**2
        x = Soff[0]/n2
        new_a = []
        new_b = []
        if i == 2:
            p = v[0]*x
            new_a = [Sd[0] - p, Sd[1] + p]
            new_b = [x]
        else:
            new_a.append(Sd[0] + x*v[0]*v[1])
            n3 = n2 + v[1]**2
            new_b.append(x*sqrt(n3))
            y = Soff[1]/n3
            for j in range(1, i - 2):
                new_a.append(Sd[j] + v[j]*(v[j + 1]*y - v[j - 1]*x))
                n1, n2 = n2, n3
                n3 = n3 + v[j + 1]**2
                new_b.append(sqrt(n1*n3)*y)
                x = y
                y = Soff[j + 1]/n3
            new_a.append(Sd[i - 2] - v[i - 2]*(v[i - 3]*x + y))
            new_b.append(sqrt(n2)*y)
            new_a.append(Sd[i - 1] + v[i - 2]*y)
        a, b = new_a, new_b
        for j in range(i - 1):
            w[j] = beta[i - 2]*w[j]/(Lambda[i - 1] - Lambda[j])
        w.append(1.0)
        sum_sq = sum(val**2 for val in w)
        u = sqrt(sum_sq)
    T = zeros(n) 
    for i in range(n):
        T[i, i] = a[i]
    for i in range(n - 1):
        T[i, i + 1] = T[i + 1, i] = b[i]
    return T